El script debe realizar un agrupamiento o Clustering con AgglomerativeClustering y distancia de Gower

    1. Usar la matriz de distancias de Gower calculada en el Ejercicio 3.

    2. Aplicar AgglomerativeClustering con metric="precomputed" y linkage="average".

    3. Determinar un número adecuado de clusters usando **Coeficiente de Silhouette**.

## Paso 1: Calcular la matriz de distancias de Gower

Reutilizamos el código del Ejercicio 3 para calcular la matriz de Gower con una muestra del dataset.


In [1]:
import sys
sys.path.append('..')
from utils import cargar_dataset_adult
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Configuración para mejor visualización en Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

# Cargar dataset
df = cargar_dataset_adult()

# Tomar muestra (5000 registros para eficiencia computacional)
sample_size = 5000
df_sample = df.sample(n=sample_size, random_state=42).copy()

print("="*60)
print("CÁLCULO DE MATRIZ DE DISTANCIAS DE GOWER")
print("="*60)
print(f"\nMuestra utilizada: {sample_size} observaciones")

# Identificar tipos de variables (sin fnlwgt)
numeric_cols = ['age', 'capital-gain', 'capital-loss', 'hours-per-week', 'education-num']
ordinal_cols = ['education']
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

# Manejo de la Variable Ordinal 'education'
education_map = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
df['education_rank'] = df['education'].map(education_map)
min_rank = df['education_rank'].min()
max_rank = df['education_rank'].max()
ordinal_rank_range = max_rank - min_rank
df_sample['education'] = (df_sample['education'].map(education_map) - min_rank) / ordinal_rank_range

# Pre-calcular los rangos para las variables numéricas
ranges = {col: df[col].max() - df[col].min() for col in numeric_cols}

# Cálculo Vectorizado de la Matriz de Distancias de Gower
print("\nCalculando matriz de distancias de Gower...")

# Separar los datos en arrays de NumPy por tipo
df_num = df_sample[numeric_cols].to_numpy(dtype=np.float64)
df_cat = df_sample[categorical_cols].to_numpy(dtype=object)
df_ord = df_sample[ordinal_cols].to_numpy(dtype=np.float64)

# Calcular la disimilitud para cada tipo de variable
numeric_dissim = np.sum([
    np.abs(np.subtract.outer(df_num[:, i], df_num[:, i])) / ranges[col]
    for i, col in enumerate(numeric_cols) if ranges[col] > 0
], axis=0)

categorical_dissim = np.sum([
    np.not_equal.outer(df_cat[:, i], df_cat[:, i]).astype(np.float64)
    for i in range(df_cat.shape[1])
], axis=0)

ordinal_dissim = np.sum([
    np.abs(np.subtract.outer(df_ord[:, i], df_ord[:, i]))
    for i in range(df_ord.shape[1])
], axis=0)

# Sumar todas las disimilitudes (numerador de Gower)
total_dissimilarity = numeric_dissim + categorical_dissim + ordinal_dissim

# Calcular los pesos (denominador de Gower) de forma vectorizada
is_not_nan = ~df_sample.drop(columns=['education_rank'], errors='ignore').isna()
weight_sum = np.dot(is_not_nan.astype(int), is_not_nan.astype(int).T)

# Calcular la matriz de Gower final, evitando la división por cero
with np.errstate(divide='ignore', invalid='ignore'):
    gower_matrix = np.where(weight_sum > 0, total_dissimilarity / weight_sum, 0)

# Asegurar que la diagonal sea exactamente 0 (distancia de un punto consigo mismo)
np.fill_diagonal(gower_matrix, 0)

print(f"\n✓ Matriz de Gower calculada: {gower_matrix.shape}")
print(f"  Rango de distancias: [{gower_matrix.min():.4f}, {gower_matrix.max():.4f}]")
print(f"  Distancia media: {gower_matrix.mean():.4f}")


CÁLCULO DE MATRIZ DE DISTANCIAS DE GOWER

Muestra utilizada: 5000 observaciones

Calculando matriz de distancias de Gower...

✓ Matriz de Gower calculada: (5000, 5000)
  Rango de distancias: [0.0000, 0.9128]
  Distancia media: 0.3295


## Paso 2: Aplicar AgglomerativeClustering con distintos valores de k

Usamos la matriz de distancias precalculada con `metric="precomputed"` y `linkage="average"`.


In [2]:
# Rango de k a probar
k_range = range(2, 11)

# Listas para almacenar resultados
silhouette_scores = []
clustering_models = {}

print("="*60)
print("APLICANDO AGGLOMERATIVE CLUSTERING CON DISTINTOS K")
print("="*60)
print("\nConfiguración: metric='precomputed', linkage='average'")
print("\nCalculando Coeficiente de Silhouette...\n")
print(" Silhouette mide la calidad del clustering considerando:")
print("   • Cohesión: qué tan cerca están los puntos de su propio cluster")
print("   • Separación: qué tan lejos están de otros clusters")
print("   • Valores: [-1, 1] → Maximizar (cercano a 1 es mejor)\n")

# Aplicar AgglomerativeClustering para cada valor de k
for k in k_range:
    # Entrenar AgglomerativeClustering
    agg = AgglomerativeClustering(
        n_clusters=k,
        metric='precomputed',
        linkage='average'
    )
    labels = agg.fit_predict(gower_matrix)
    
    # Guardar el modelo
    clustering_models[k] = {'model': agg, 'labels': labels}
    
    # Calcular Coeficiente de Silhouette
    # IMPORTANTE: Para Silhouette con distancias precalculadas usamos metric='precomputed'
    silhouette = silhouette_score(gower_matrix, labels, metric='precomputed')
    silhouette_scores.append(silhouette)
    
    print(f"k={k:2d} | Silhouette: {silhouette:.4f}")

print("\n✓ Entrenamiento completado para todos los valores de k")


APLICANDO AGGLOMERATIVE CLUSTERING CON DISTINTOS K

Configuración: metric='precomputed', linkage='average'

Calculando Coeficiente de Silhouette...

 Silhouette mide la calidad del clustering considerando:
   • Cohesión: qué tan cerca están los puntos de su propio cluster
   • Separación: qué tan lejos están de otros clusters
   • Valores: [-1, 1] → Maximizar (cercano a 1 es mejor)

k= 2 | Silhouette: 0.4034
k= 3 | Silhouette: 0.3595
k= 4 | Silhouette: 0.3343
k= 5 | Silhouette: 0.2681
k= 6 | Silhouette: 0.1842
k= 7 | Silhouette: 0.1663
k= 8 | Silhouette: 0.1652
k= 9 | Silhouette: 0.1428
k=10 | Silhouette: 0.1305

✓ Entrenamiento completado para todos los valores de k


## Paso 3: Determinar el número óptimo de clusters

Usamos el **Coeficiente de Silhouette** para determinar el k óptimo:

- **Objetivo**: Maximizar (valores cercanos a 1 son mejores)
- **Ventaja con Gower**: Funciona directamente con matrices de distancias precalculadas
- **Interpretación**: Mide la cohesión interna y separación externa de los clusters


In [3]:
# Determinar k óptimo según Silhouette (maximizar)
k_optimal = k_range[np.argmax(silhouette_scores)]
max_silhouette = max(silhouette_scores)

# Resumen de resultados
print("="*60)
print("DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS")
print("="*60)

# Tabla resumen
results_df = pd.DataFrame({
    'k': list(k_range),
    'Silhouette': silhouette_scores
})

print(f"\nTabla Resumen de Coeficiente de Silhouette:")
print(results_df.to_string(index=False))

# Marcar el óptimo
print(f"\n{'='*60}")
print(f"✓ K ÓPTIMO = {k_optimal}")
print(f"{'='*60}")
print(f"\nCoeficiente de Silhouette: {max_silhouette:.4f}")
print(f"\nInterpretación:")
if max_silhouette > 0.7:
    print("   → Estructura de clustering FUERTE")
elif max_silhouette > 0.5:
    print("   → Estructura de clustering RAZONABLE")
elif max_silhouette > 0.25:
    print("   → Estructura de clustering DÉBIL pero presente")
else:
    print("   → Estructura de clustering ARTIFICIAL (no hay clusters naturales claros)")


DETERMINACIÓN DEL NÚMERO ÓPTIMO DE CLUSTERS

Tabla Resumen de Coeficiente de Silhouette:
 k  Silhouette
 2      0.4034
 3      0.3595
 4      0.3343
 5      0.2681
 6      0.1842
 7      0.1663
 8      0.1652
 9      0.1428
10      0.1305

✓ K ÓPTIMO = 2

Coeficiente de Silhouette: 0.4034

Interpretación:
   → Estructura de clustering DÉBIL pero presente


## Paso 4: Análisis del modelo óptimo

Analizamos los clusters usando el k óptimo determinado por las métricas.


In [4]:
# Usar el k óptimo según Silhouette
labels_optimal = clustering_models[k_optimal]['labels']

# Recuperar datos originales (antes de transformar education)
df_original = df.sample(n=sample_size, random_state=42)
all_cols = numeric_cols + categorical_cols
df_clustered = df_original[all_cols].copy()
df_clustered['Cluster'] = labels_optimal

print("="*60)
print(f"ANÁLISIS DEL MODELO ÓPTIMO (k={k_optimal})")
print("="*60)

# Estadísticas por cluster
print(f"\nDistribución de observaciones por cluster:")
cluster_counts = pd.Series(labels_optimal).value_counts().sort_index()
for cluster_id, count in cluster_counts.items():
    percentage = (count / len(labels_optimal)) * 100
    print(f"   Cluster {cluster_id}: {count:6d} observaciones ({percentage:5.2f}%)")

# Características promedio por cluster (solo numéricas)
print(f"\n\nCaracterísticas numéricas promedio por cluster:\n")
cluster_summary = df_clustered.groupby('Cluster')[numeric_cols].mean()
print(cluster_summary.round(2))

# Distribución de variables categóricas importantes por cluster
print(f"\n\nCaracterísticas categóricas más frecuentes por cluster:")
for cluster_id in sorted(cluster_summary.index):
    print(f"\n   CLUSTER {cluster_id}:")
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster_id]
    
    # Variables categóricas clave
    for cat_col in ['workclass', 'education', 'marital-status', 'income']:
        if cat_col in df_clustered.columns:
            mode_val = cluster_data[cat_col].mode()[0] if len(cluster_data[cat_col].mode()) > 0 else 'N/A'
            freq = (cluster_data[cat_col] == mode_val).sum() / len(cluster_data) * 100
            print(f"   • {cat_col}: {mode_val} ({freq:.1f}%)")

print(f"\n\n✓ Análisis completado con k óptimo = {k_optimal}")


ANÁLISIS DEL MODELO ÓPTIMO (k=2)

Distribución de observaciones por cluster:
   Cluster 0:      6 observaciones ( 0.12%)
   Cluster 1:   4994 observaciones (99.88%)


Características numéricas promedio por cluster:

           age  capital-gain  capital-loss  hours-per-week  education-num
Cluster                                                                  
0        55.33        577.33        715.33           30.50          12.83
1        38.47       1059.43         79.42           40.49          10.09


Características categóricas más frecuentes por cluster:

   CLUSTER 0:
   • workclass: Self-emp-not-inc (16.7%)
   • marital-status: Married-civ-spouse (66.7%)
   • income: <=50K (100.0%)

   CLUSTER 1:
   • workclass: Private (69.8%)
   • marital-status: Married-civ-spouse (44.8%)
   • income: <=50K (76.4%)


✓ Análisis completado con k óptimo = 2


## Paso 5: Interpretación detallada de características por cluster

Comparación de las características promedio de cada cluster con la media global.


In [5]:
# Interpretación detallada
print("="*60)
print("INTERPRETACIÓN DETALLADA DE CLUSTERS")
print("="*60)

for cluster_id in sorted(cluster_summary.index):
    print(f"\n🔹 CLUSTER {cluster_id}:")
    print(f"   Tamaño: {cluster_counts[cluster_id]} observaciones ({cluster_counts[cluster_id]/len(labels_optimal)*100:.1f}%)")
    
    print(f"\n   Características numéricas:")
    for col in numeric_cols:
        val = cluster_summary.loc[cluster_id, col]
        global_mean = df_original[col].mean()
        diff_percent = ((val - global_mean) / global_mean * 100) if global_mean != 0 else 0
        
        if abs(diff_percent) > 20:  # Diferencia significativa
            direction = "↑" if diff_percent > 0 else "↓"
            print(f"   • {col}: {val:.1f} ({direction} {abs(diff_percent):.0f}% vs media global)")
        else:
            print(f"   • {col}: {val:.1f} (~ media global)")
    
    print(f"\n   Características categóricas dominantes:")
    cluster_data = df_clustered[df_clustered['Cluster'] == cluster_id]
    for cat_col in ['education', 'marital-status', 'income']:
        if cat_col in df_clustered.columns:
            mode_val = cluster_data[cat_col].mode()[0] if len(cluster_data[cat_col].mode()) > 0 else 'N/A'
            freq = (cluster_data[cat_col] == mode_val).sum() / len(cluster_data) * 100
            print(f"   • {cat_col}: {mode_val} ({freq:.1f}%)")

print("\n" + "="*60)
print("="*60)


INTERPRETACIÓN DETALLADA DE CLUSTERS

🔹 CLUSTER 0:
   Tamaño: 6 observaciones (0.1%)

   Características numéricas:
   • age: 55.3 (↑ 44% vs media global)
   • capital-gain: 577.3 (↓ 45% vs media global)
   • capital-loss: 715.3 (↑ 792% vs media global)
   • hours-per-week: 30.5 (↓ 25% vs media global)
   • education-num: 12.8 (↑ 27% vs media global)

   Características categóricas dominantes:
   • marital-status: Married-civ-spouse (66.7%)
   • income: <=50K (100.0%)

🔹 CLUSTER 1:
   Tamaño: 4994 observaciones (99.9%)

   Características numéricas:
   • age: 38.5 (~ media global)
   • capital-gain: 1059.4 (~ media global)
   • capital-loss: 79.4 (~ media global)
   • hours-per-week: 40.5 (~ media global)
   • education-num: 10.1 (~ media global)

   Características categóricas dominantes:
   • marital-status: Married-civ-spouse (44.8%)
   • income: <=50K (76.4%)

